In [2]:
from IPython.display import Image

- $L\in R^{B\times T}$ 表示 loss mat, $M\in {\{0,1\}}^{B\times T}$ 表示损失掩码 (loss_mask)（为 1 表示损失计算在内）
    - $ \mathcal{L}_{\text{token-mean}} = \frac{\sum_{i=1}^{B} \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{\sum_{i=1}^{B} \sum_{j=1}^{T} M_{i,j}} $
    - $\mathcal{L}_{\text{seq-mean-token-sum}} = \frac{1}{B} \sum_{i=1}^{B} \left( \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j} \right)$
    - $\mathcal{L}_{\text{seq-mean-token-mean}} = \frac{1}{B} \sum_{i=1}^{B} \left( \frac{\sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{\sum_{j=1}^{T} M_{i,j}} \right)$
    - $\mathcal{L}_{\text{seq-mean-token-sum-norm}} = \frac{\sum_{i=1}^{B} \sum_{j=1}^{T} L_{i,j} \cdot M_{i,j}}{T} $

```python
# "token-mean"
s = masked_sum(values, mask, axis)
s / (mask.sum(axis=axis) + 1e-8)
```
- https://verl.readthedocs.io/en/latest/algo/grpo.html
    - The original GRPO paper takes the sample-level loss (`seq-mean-token-mean`), which may be unstable in long-CoT scenarios.
    - DrGRPO: `seq-mean-token-sum-norm”`
    - DAPO：`token-mean`
        - https://verl.readthedocs.io/en/latest/algo/dapo.html#flexible-loss-aggregation-mode-token-level-loss
        - Setting `loss_agg_mode` to `token-mean` will mean the (policy gradient) loss across all the tokens in all the sequences in a mini-batch.

In [3]:
Image(url='./figs/seq-token-pg.jpg', width=500)

- PG：序列级回报 ⇒ token 级梯度 = “对所有 token 的和”。
    - $\pi_\theta(o|q)=\Pi_{t=1}^{|o|}\pi_\theta(o_t|q,o_{\lt t})$
        - $\log\pi_\theta(o|q)=\sum_{t=1}^{|o|}\pi_\theta(o_t|q,o_{\lt t})$
        - 来自概率分解，因此必须求和；
    - 最大化整个序列的期望总奖励 $\mathbb E[R(q,o)]$，梯度更新的幅度正比于总奖励。一个长序列即使总奖励很高，它的更新权重也应该很高。
- $\frac1G\sum_{i=1}^G$ 是对一个批次（batch）中的 $G$ 个样本取平均，这是标准操作。
    - 用样本的平均值来估计理论上的期望值
    - 用样本平均才是标准的 MC 估计；这不会改变最优点，只让梯度尺度与 $G$ 无关
- Vanilla GRPO 为什么是错的
    - 由于除以了序列长度 $|o_i|$，优化的目标变成了最大化每个时间步的平均奖励 $\mathbb E[\frac{R(q,o)}{|o|}]$
    - 由最大化回报 => 最大化平均每个词的奖励
- 一个case
    - 序列 A：长度为 20，质量很高，总奖励 $R_A=10$
    - 序列 B：长度为 2，有一些小错误，总奖励 $R_B=4$。
    - 根据公式一 (正确逻辑)：序列 A 的总奖励 (10) 远高于序列 B (4)，因此模型会大力度地增加生成序列 A 的概率。这是我们想要的。
    - 根据公式二 (错误逻辑)：
        - 序列 A 的平均奖励是 10/20 = 0.5
        - 序列 B 的平均奖励是 4/2 = 2
    - 在这个错误的框架下，模型会认为序列 B "更好"，因为它每个词的平均贡献更大。于是，模型会反过来去增加生成序列 B 的概率，而抑制序列 A。
        - 系统性地偏向短输出（